# Pipeline Huấn luyện & Đánh giá — ShieldAI (PhoBERT + MLP)

Notebook **duy nhất** gom toàn bộ quy trình huấn luyện và đánh giá:

| Phần | Nội dung |
|------|----------|
| **0** | Phương pháp PhoBERT & cấu hình (lý thuyết + minh họa) |
| **1** | Tiền xử lý & lọc dữ liệu (`dataset_cleaner`) |
| **2** | Nhúng PhoBERT-base → lưu `.npy` |
| **3** | Kiểm tra mẫu dữ liệu |
| **4** | Huấn luyện MLP text-only → lưu `.joblib` |
| **5** | Kết quả thực nghiệm (confusion matrix, ROC/AUC, 5-fold CV) |

> `REGENERATE_EMBEDDINGS = False` nếu đã có `phobert_base_features.npy` (tránh chạy lại ~1 giờ trên CPU).

In [ ]:
# !pip install pyvi transformers torch pandas numpy seaborn matplotlib tqdm scikit-learn joblib

import joblib
from pathlib import Path
import sys

# Đảm bảo import được package `backend/` dù Jupyter chạy ở thư mục nào
_cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [_cwd, *_cwd.parents] if (p / "backend" / "__init__.py").exists()), _cwd)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"PROJECT_ROOT = {PROJECT_ROOT}")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from pyvi import ViTokenizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer

from backend.dataset_cleaner import prepare_training_dataset, add_segmented_column
from backend.training.merge_last_datasets import save_merged_dataset

REBUILD_MERGED_DATASET = True  # True = gộp lại toàn bộ dữ liệu trong last/
REGENERATE_EMBEDDINGS = False  # True = chạy lại embedding PhoBERT
RUN_EXPERIMENTS = True  # True = chạy lại bảng metric/biểu đồ sau khi train

FEATURES_PATH = PROJECT_ROOT / 'backend/models/phobert_merged_features.npy'
LABELS_PATH = PROJECT_ROOT / 'backend/models/phobert_merged_labels.npy'
LAST_DATA_DIR = PROJECT_ROOT / 'last'
DATA_PATH = PROJECT_ROOT / 'backend/data/merged_fake_news_dataset.csv'

np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


## Phần 0 — Phương pháp PhoBERT & cấu hình

### 0.1. Tiếp cận text-only
Sử dụng **embedding PhoBERT** (768 chiều) + **MLP** để phân loại tin thật / tin giả.

### 0.2. Kiến trúc & hyperparameters (tham khảo fine-tune)
Đề tài dùng **PhoBERT-base frozen** + MLP; bảng dưới mô tả cấu hình nếu fine-tune end-to-end:

| Thông số | Giá trị điển hình |
|----------|-------------------|
| Model | `vinai/phobert-base` (12 layers, 768 hidden) |
| Learning rate | `2e-5` |
| Batch size | 16 |
| Epochs | 5 |
| Optimizer | AdamW |
| LR scheduler | linear |

In [ ]:
from transformers import AutoConfig, AutoModelForSequenceClassification, TrainingArguments

model_name = 'vinai/phobert-base'
config = AutoConfig.from_pretrained(model_name)
print('==== KIẾN TRÚC PHOBERT-BASE ====')
print(f'- Hidden layers: {config.num_hidden_layers}')
print(f'- Hidden size: {config.hidden_size}')
print(f'- Attention heads: {config.num_attention_heads}')
print(f'- Vocab size: {config.vocab_size}')

_ = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
print('[+] Khởi tạo PhoBERT cho classification (2 nhãn) — tham khảo kiến trúc.')

training_args = TrainingArguments(
    output_dir='./results_fake_news',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    optim='adamw_torch',
    lr_scheduler_type='linear',
    seed=42,
    eval_strategy='epoch',
    save_strategy='epoch',
)
print('\n==== HYPERPARAMETERS (fine-tune tham khảo) ====')
print(f'LR={training_args.learning_rate}, batch={training_args.per_device_train_batch_size}, epochs={training_args.num_train_epochs}')


In [ ]:
from transformers import AutoTokenizer

demo_tokenizer = AutoTokenizer.from_pretrained('vinai/phobert-base')

def phobert_preprocess(text: str):
    segmented = ViTokenizer.tokenize(text)
    encoded = demo_tokenizer(
        segmented, padding='max_length', truncation=True, max_length=256, return_tensors='pt'
    )
    return segmented, encoded

raw_news = 'Công an thành phố Hà Nội vừa triệt phá một đường dây tung tin đồn thất thiệt trên mạng xã hội.'
segmented_news, bpe_encoded = phobert_preprocess(raw_news)
print('[+] Văn bản gốc:', raw_news)
print('[+] Tách từ (PyVi):', segmented_news)
print('[+] Token IDs (15 đầu):', bpe_encoded['input_ids'][0].tolist()[:15])


## Phần 1 — Nạp và lọc dữ liệu

In [ ]:
if REBUILD_MERGED_DATASET or not DATA_PATH.exists():
    save_merged_dataset(input_dir=LAST_DATA_DIR, output_path=DATA_PATH)

df = prepare_training_dataset(DATA_PATH, add_segmented=False)
print(f'Tổng số tin bài sau lọc: {len(df)}')
print('\nPhân bố nhãn:')
print(df['is_fake'].value_counts().rename(index={False: 'real', True: 'fake'}))
print('\nPhân bố theo dataset:')
display(pd.crosstab(df['dataset_name'], df['is_fake'], margins=True))


## Phần 2 — PhoBERT: Tách từ & Nhúng (Embedding)

In [ ]:
need_embed = REGENERATE_EMBEDDINGS or not (FEATURES_PATH.exists() and LABELS_PATH.exists())

if need_embed:
    print('Đang tách từ...')
    df = add_segmented_column(df)
    print(df[['content', 'content_segmented']].head(2))
else:
    print('Bỏ qua embedding — dùng file .npy có sẵn.')


In [ ]:
if need_embed:
    print('Đang tải PhoBERT-base...')
    tokenizer = AutoTokenizer.from_pretrained('vinai/phobert-base')
    model = AutoModel.from_pretrained('vinai/phobert-base', output_attentions=True).to(device)
    model.eval()

    def get_phobert_embeddings(texts, batch_size=32, max_len=256):
        all_embeddings = []
        with torch.no_grad():
            for i in tqdm(range(0, len(texts), batch_size), desc='Embedding'):
                batch = texts[i:i + batch_size]
                encoded = tokenizer(batch, padding=True, truncation=True, max_length=max_len, return_tensors='pt')
                input_ids = encoded['input_ids'].to(device)
                attention_mask = encoded['attention_mask'].to(device)
                outputs = model(input_ids, attention_mask=attention_mask)
                all_embeddings.append(outputs.last_hidden_state[:, 0, :].cpu().numpy())
        return np.vstack(all_embeddings)

    vectors = get_phobert_embeddings(df['content_segmented'].tolist())
    print(f'Kích thước embedding: {vectors.shape}')
    np.save(FEATURES_PATH, vectors)
    np.save(LABELS_PATH, df['is_fake'].values)
    print(f'Đã lưu: {FEATURES_PATH.name}, {LABELS_PATH.name}')


### (Tùy chọn) Trực quan Attention

In [ ]:
# Chỉ chạy khi vừa embed (model + tokenizer đã load)
if need_embed:
    sample = 'Mạng xã hội chấn_động khi xuất_hiện tin_đồn nam ca_sĩ nổi_tiếng đột_tử trong đêm.'
    inputs = tokenizer(sample, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    attention = outputs.attentions[-1][0, 0].cpu().numpy()
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    plt.figure(figsize=(10, 8))
    sns.heatmap(attention, xticklabels=tokens, yticklabels=tokens, cmap='YlOrRd')
    plt.title('Self-Attention PhoBERT (minh họa)')
    plt.show()


## Phần 3 — Kiểm tra mẫu dữ liệu

Xem nhanh vài mẫu trước khi huấn luyện MLP.

In [ ]:
print('Kiểm tra mẫu dữ liệu text-only.')
df[['is_fake', 'title', 'content']].sample(5)

## Phần 4 — Huấn luyện PhoBERT MLP (text-only)

In [ ]:
X_phobert = np.load(FEATURES_PATH)
y_labels = np.load(LABELS_PATH)
assert len(X_phobert) == len(y_labels), 'Số dòng feature/label không khớp'
assert len(X_phobert) == len(df), 'Embedding cache không khớp dataset hiện tại. Hãy đặt REGENERATE_EMBEDDINGS = True.'

print(f'PhoBERT features: {X_phobert.shape}')

X_train, X_test, y_train, y_test = train_test_split(
    X_phobert, y_labels, test_size=0.3, random_state=42, stratify=y_labels
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
print('Huấn luyện MLP Classifier...')
mlp_model = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    max_iter=500,
    alpha=0.1,
    early_stopping=True,
    random_state=42,
)
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)
try:
    mlp_model.fit(X_train_scaled, y_train, sample_weight=sample_weights)
except TypeError:
    print('Cảnh báo: scikit-learn hiện tại không hỗ trợ sample_weight cho MLPClassifier; train không cân bằng trọng số.')
    mlp_model.fit(X_train_scaled, y_train)

y_pred = mlp_model.predict(X_test_scaled)
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(classification_report(y_test, y_pred))

MODELS_DIR = PROJECT_ROOT / 'backend/models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(mlp_model, MODELS_DIR / 'phobert_mlp_model.joblib')
joblib.dump(scaler, MODELS_DIR / 'phobert_scaler.joblib')
print('Đã lưu: phobert_mlp_model.joblib, phobert_scaler.joblib')

## Phần 5 — Kết quả thực nghiệm

Đánh giá mô hình **PhoBERT text-only**: confusion matrix, ROC/AUC và 5-fold CV.

Script: `backend/experiments/run_experimental_evaluation.py`

In [ ]:
from IPython.display import Image, display

EXPERIMENTS_DIR = PROJECT_ROOT / 'backend/experiments'
FIG_DIR = EXPERIMENTS_DIR / 'figures' / 'experimental'
FIG_DIR.mkdir(parents=True, exist_ok=True)

if RUN_EXPERIMENTS or not (FIG_DIR / 'cv_summary.csv').exists():
    print('Chạy đánh giá thực nghiệm...')
    from backend.experiments.run_experimental_evaluation import main as run_experiments
    run_experiments()
else:
    print('Dùng kết quả có sẵn tại', FIG_DIR)

df_metrics = pd.read_csv(FIG_DIR / 'metrics_summary.csv')
df_cv = pd.read_csv(FIG_DIR / 'cv_summary.csv')
display(df_metrics)
df_cv

### 5.1. Confusion matrix

In [ ]:
img = plt.imread(FIG_DIR / 'confusion_matrix_text_only.png')
plt.figure(figsize=(6, 5))
plt.imshow(img)
plt.axis('off')
plt.title('Confusion Matrix — PhoBERT text-only (tập kiểm thử 30%)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.2. Bảng chỉ số hold-out (30%)

In [ ]:
display(df_metrics)

### 5.3. ROC / AUC và 5-fold CV

In [ ]:
display(Image(filename=str(FIG_DIR / 'roc_curves.png')))
display(df_cv)